# Car Insurance Premium Prediction ML Pipeline

This notebook implements an end-to-end ML pipeline for predicting car insurance premiums using:
- **Snowflake Feature Store** (with Online Feature Store for real-time serving)
- **XGBoost** model with StandardScaler preprocessing
- **Snowflake Model Registry** for model management
- **SPCS Inference Service** for real-time predictions

## 1. Setup and Configuration

In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import StructType, StructField, StringType, IntegerType, FloatType, DateType

session = Session.builder.config(
    "connection_name",
    os.getenv("SNOWFLAKE_CONNECTION_NAME") or "talent_keypair"
).create()



In [2]:
DATABASE = "CC_ML_INSURANCE"
SCHEMA = "CAR_PRICING"

session.sql(f'CREATE OR REPLACE DATABASE {DATABASE}').collect()
session.sql(f'USE DATABASE {DATABASE}').collect()
session.sql(f'CREATE OR REPLACE SCHEMA {SCHEMA}').collect()
session.sql(f'USE SCHEMA {SCHEMA}').collect()


session.use_database(DATABASE)
session.use_schema(SCHEMA)

print(f"Connected to: {session.get_current_account()}")
print(f"Using: {DATABASE}.{SCHEMA}")

Connected to: "phb14991"
Using: CC_ML_INSURANCE.CAR_PRICING


## 2. Generate Synthetic Car Insurance Data

In [3]:
np.random.seed(42)
random.seed(42)

N_CUSTOMERS = 5000
N_POLICIES = 8000

CAR_MAKES = {
    'Toyota': ['Camry', 'Corolla', 'RAV4', 'Highlander', 'Prius'],
    'Honda': ['Civic', 'Accord', 'CR-V', 'Pilot', 'Odyssey'],
    'Ford': ['F-150', 'Mustang', 'Explorer', 'Escape', 'Bronco'],
    'BMW': ['3 Series', '5 Series', 'X3', 'X5', 'M3'],
    'Mercedes': ['C-Class', 'E-Class', 'GLC', 'GLE', 'S-Class'],
    'Chevrolet': ['Silverado', 'Malibu', 'Equinox', 'Tahoe', 'Corvette'],
    'Tesla': ['Model 3', 'Model Y', 'Model S', 'Model X'],
    'Nissan': ['Altima', 'Rogue', 'Sentra', 'Pathfinder', 'Maxima']
}

COLORS = ['Black', 'White', 'Silver', 'Gray', 'Blue', 'Red', 'Green', 'Brown']
FUEL_TYPES = ['Gasoline', 'Diesel', 'Hybrid', 'Electric']
TRANSMISSIONS = ['Automatic', 'Manual', 'CVT']
COVERAGE_TYPES = ['Basic', 'Standard', 'Premium', 'Comprehensive']

BASE_PRICES = {
    'Toyota': 28000, 'Honda': 27000, 'Ford': 35000, 'BMW': 55000,
    'Mercedes': 60000, 'Chevrolet': 32000, 'Tesla': 50000, 'Nissan': 26000
}

print("Data constants defined")

Data constants defined


In [4]:
customer_ids = [f"CUST_{str(i).zfill(6)}" for i in range(1, N_CUSTOMERS + 1)]
first_names = ['James', 'Mary', 'John', 'Patricia', 'Robert', 'Jennifer', 'Michael', 'Linda', 
               'William', 'Elizabeth', 'David', 'Susan', 'Richard', 'Jessica', 'Joseph', 'Sarah']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 'Davis',
              'Rodriguez', 'Martinez', 'Wilson', 'Anderson', 'Taylor', 'Thomas', 'Moore', 'Jackson']

customers_data = []
for cust_id in customer_ids:
    age = int(np.random.normal(42, 15))
    age = max(18, min(80, age))
    years_licensed = min(age - 16, int(np.random.exponential(15)))
    years_licensed = max(1, years_licensed)
    claims_history = int(np.random.exponential(0.8))
    claims_history = min(claims_history, 10)
    credit_score = int(np.random.normal(700, 80))
    credit_score = max(300, min(850, credit_score))
    
    customers_data.append({
        'CUSTOMER_ID': cust_id,
        'FIRST_NAME': random.choice(first_names),
        'LAST_NAME': random.choice(last_names),
        'AGE': age,
        'GENDER': random.choice(['M', 'F']),
        'YEARS_LICENSED': years_licensed,
        'CLAIMS_HISTORY': claims_history,
        'CREDIT_SCORE': credit_score,
        'STATE': random.choice(['CA', 'TX', 'FL', 'NY', 'IL', 'PA', 'OH', 'GA', 'NC', 'MI'])
    })

customers_df = pd.DataFrame(customers_data)
print(f"Generated {len(customers_df)} customers")
customers_df.head()

Generated 5000 customers


,CUSTOMER_ID,FIRST_NAME,LAST_NAME,AGE,GENDER,YEARS_LICENSED,CLAIMS_HISTORY,CREDIT_SCORE,STATE
0,CUST_000001,Patricia,Smith,49,F,19,0,688,NY
1,CUST_000002,Linda,Jones,38,M,1,1,681,NC
2,CUST_000003,John,Thomas,65,M,1,2,761,CA
3,CUST_000004,John,Miller,34,M,3,0,743,NC
4,CUST_000005,James,Miller,45,F,8,0,546,NY


In [5]:
current_year = datetime.now().year

policies_data = []
for i in range(N_POLICIES):
    cust_id = random.choice(customer_ids)
    customer = customers_df[customers_df['CUSTOMER_ID'] == cust_id].iloc[0]
    
    car_make = random.choice(list(CAR_MAKES.keys()))
    car_model = random.choice(CAR_MAKES[car_make])
    car_year = random.randint(2010, current_year)
    car_age = current_year - car_year
    
    if car_make == 'Tesla':
        fuel_type = 'Electric'
    elif car_make in ['BMW', 'Mercedes']:
        fuel_type = random.choices(FUEL_TYPES, weights=[0.6, 0.15, 0.2, 0.05])[0]
    else:
        fuel_type = random.choices(FUEL_TYPES, weights=[0.7, 0.1, 0.15, 0.05])[0]
    
    transmission = random.choices(TRANSMISSIONS, weights=[0.7, 0.15, 0.15])[0]
    avg_km_per_year = np.random.normal(15000, 5000)
    kilometers = int(max(1000, car_age * avg_km_per_year + np.random.normal(0, 5000)))
    engine_size = random.choice([1.5, 1.8, 2.0, 2.4, 2.5, 3.0, 3.5, 4.0, 5.0])
    
    base_price = BASE_PRICES[car_make]
    depreciation = 0.85 ** car_age
    km_factor = max(0.5, 1 - (kilometers / 300000))
    car_value = base_price * depreciation * km_factor
    
    coverage_type = random.choice(COVERAGE_TYPES)
    deductible = random.choice([250, 500, 1000, 1500, 2000])
    
    base_premium = 800
    
    if customer['AGE'] < 25:
        age_factor = 1.5
    elif customer['AGE'] > 65:
        age_factor = 1.2
    else:
        age_factor = 1.0
    
    experience_factor = max(0.8, 1.3 - (customer['YEARS_LICENSED'] * 0.02))
    claims_factor = 1 + (customer['CLAIMS_HISTORY'] * 0.15)
    credit_factor = max(0.8, 1.3 - ((customer['CREDIT_SCORE'] - 600) / 500))
    
    car_value_factor = 0.03 * (car_value / 10000)
    make_factor = {'BMW': 1.3, 'Mercedes': 1.35, 'Tesla': 1.25, 'Chevrolet': 1.05,
                   'Ford': 1.1, 'Toyota': 0.95, 'Honda': 0.95, 'Nissan': 1.0}.get(car_make, 1.0)
    
    coverage_factor = {'Basic': 0.7, 'Standard': 1.0, 'Premium': 1.3, 'Comprehensive': 1.6}[coverage_type]
    deductible_factor = {250: 1.2, 500: 1.1, 1000: 1.0, 1500: 0.9, 2000: 0.85}[deductible]
    
    annual_premium = base_premium * age_factor * experience_factor * claims_factor * credit_factor
    annual_premium = annual_premium * (1 + car_value_factor) * make_factor
    annual_premium = annual_premium * coverage_factor * deductible_factor
    annual_premium = annual_premium + np.random.normal(0, 50)
    annual_premium = max(400, min(5000, annual_premium))
    
    policy_id = f"POL_{str(i+1).zfill(7)}"
    start_date = datetime.now() - timedelta(days=random.randint(0, 365))
    
    policies_data.append({
        'POLICY_ID': policy_id,
        'CUSTOMER_ID': cust_id,
        'CAR_MAKE': car_make,
        'CAR_MODEL': car_model,
        'CAR_YEAR': car_year,
        'COLOR': random.choice(COLORS),
        'KILOMETERS': kilometers,
        'ENGINE_SIZE': engine_size,
        'FUEL_TYPE': fuel_type,
        'TRANSMISSION': transmission,
        'COVERAGE_TYPE': coverage_type,
        'DEDUCTIBLE': deductible,
        'ESTIMATED_CAR_VALUE': round(car_value, 2),
        'ANNUAL_PREMIUM': round(annual_premium, 2),
        'POLICY_START_DATE': start_date.strftime('%Y-%m-%d'),
        'UPDATED_AT': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    })

policies_df = pd.DataFrame(policies_data)
print(f"Generated {len(policies_df)} policies")
print(f"Premium range: ${policies_df['ANNUAL_PREMIUM'].min():.2f} - ${policies_df['ANNUAL_PREMIUM'].max():.2f}")
print(f"Mean premium: ${policies_df['ANNUAL_PREMIUM'].mean():.2f}")
policies_df.head()

Generated 8000 policies
Premium range: $400.00 - $5000.00
Mean premium: $1475.00


,POLICY_ID,CUSTOMER_ID,CAR_MAKE,CAR_MODEL,CAR_YEAR,COLOR,KILOMETERS,ENGINE_SIZE,FUEL_TYPE,TRANSMISSION,COVERAGE_TYPE,DEDUCTIBLE,ESTIMATED_CAR_VALUE,ANNUAL_PREMIUM,POLICY_START_DATE,UPDATED_AT
0,POL_0000001,CUST_004018,BMW,3 Series,2018,Blue,95681,3.5,Gasoline,Automatic,Standard,2000,10207.08,1269.55,2025-12-08,2026-01-29 08:10:55
1,POL_0000002,CUST_002124,Mercedes,GLE,2013,Green,82714,3.0,Gasoline,Automatic,Standard,250,5254.21,1462.14,2025-04-10,2026-01-29 08:10:55
2,POL_0000003,CUST_004712,Toyota,Highlander,2025,Blue,3932,1.5,Gasoline,Automatic,Standard,500,23488.06,1084.45,2025-05-16,2026-01-29 08:10:55
3,POL_0000004,CUST_004171,Nissan,Sentra,2020,Silver,107873,2.4,Diesel,Automatic,Premium,2000,6279.92,1350.70,2025-06-11,2026-01-29 08:10:55
4,POL_0000005,CUST_004022,Honda,Pilot,2018,Blue,154701,3.5,Hybrid,Automatic,Basic,250,3678.62,807.71,2025-07-17,2026-01-29 08:10:55


In [6]:
session.write_pandas(customers_df, "CUSTOMERS", auto_create_table=True, overwrite=True)
session.write_pandas(policies_df, "POLICIES", auto_create_table=True, overwrite=True)

print("Data uploaded to Snowflake:")
print(f"  - {DATABASE}.{SCHEMA}.CUSTOMERS: {session.table('CUSTOMERS').count()} rows")
print(f"  - {DATABASE}.{SCHEMA}.POLICIES: {session.table('POLICIES').count()} rows")

Data uploaded to Snowflake:
  - CC_ML_INSURANCE.CAR_PRICING.CUSTOMERS: 5000 rows
  - CC_ML_INSURANCE.CAR_PRICING.POLICIES: 8000 rows


## 3. Create Feature Store with Online Features

In [7]:
from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode
from snowflake.ml.feature_store.feature_view import OnlineConfig

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=SCHEMA,
    default_warehouse='COMPUTE_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

print(f"Feature Store created: {DATABASE}.{SCHEMA}")

Feature Store created: CC_ML_INSURANCE.CAR_PRICING


In [8]:
customer_entity = Entity(
    name="CUSTOMER",
    join_keys=["CUSTOMER_ID"],
    desc="Customer entity for car insurance"
)

fs.register_entity(customer_entity)
print("Entity 'CUSTOMER' registered")

fs.list_entities().to_pandas()

Entity 'CUSTOMER' registered


,NAME,JOIN_KEYS,DESC,OWNER
0,CUSTOMER,"[""CUSTOMER_ID""]",Customer entity for car insurance,SPCS_PSE_ROLE


In [9]:
current_year_val = datetime.now().year

customer_features_sql = f"""
SELECT 
    c.CUSTOMER_ID,
    c.AGE,
    c.YEARS_LICENSED,
    c.CLAIMS_HISTORY,
    c.CREDIT_SCORE,
    
    -- Risk Score: Higher for young/old drivers, many claims, low credit
    ROUND(
        (CASE WHEN c.AGE < 25 THEN 30 WHEN c.AGE > 65 THEN 20 ELSE 0 END) +
        (c.CLAIMS_HISTORY * 15) +
        (GREATEST(0, 40 - c.YEARS_LICENSED)) +
        (GREATEST(0, (700 - c.CREDIT_SCORE) / 10))
    , 2) AS RISK_SCORE,
    
    -- Average claims per year licensed
    ROUND(c.CLAIMS_HISTORY / GREATEST(1, c.YEARS_LICENSED), 4) AS AVG_CLAIMS_PER_YEAR,
    
    -- Credit tier
    CASE 
        WHEN c.CREDIT_SCORE >= 750 THEN 'Excellent'
        WHEN c.CREDIT_SCORE >= 700 THEN 'Good'
        WHEN c.CREDIT_SCORE >= 650 THEN 'Fair'
        ELSE 'Poor'
    END AS CREDIT_TIER,
    
    -- Total policies
    COUNT(p.POLICY_ID) AS TOTAL_POLICIES,
    
    -- Average car age across all policies
    ROUND(AVG({current_year_val} - p.CAR_YEAR), 2) AS AVG_CAR_AGE,
    
    -- Average kilometers across all cars
    ROUND(AVG(p.KILOMETERS), 0) AS AVG_KILOMETERS,
    
    -- Total estimated car value
    ROUND(SUM(p.ESTIMATED_CAR_VALUE), 2) AS TOTAL_CAR_VALUE,
    
    -- Average deductible chosen by customer
    ROUND(AVG(p.DEDUCTIBLE), 0) AS AVG_DEDUCTIBLE,
    
    CURRENT_TIMESTAMP() AS UPDATED_AT
    
FROM {DATABASE}.{SCHEMA}.CUSTOMERS c
LEFT JOIN {DATABASE}.{SCHEMA}.POLICIES p ON c.CUSTOMER_ID = p.CUSTOMER_ID
GROUP BY c.CUSTOMER_ID, c.AGE, c.YEARS_LICENSED, c.CLAIMS_HISTORY, c.CREDIT_SCORE
"""

customer_features_df = session.sql(customer_features_sql)
customer_features_df.show(5)

-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"AGE"  |"YEARS_LICENSED"  |"CLAIMS_HISTORY"  |"CREDIT_SCORE"  |"RISK_SCORE"  |"AVG_CLAIMS_PER_YEAR"  |"CREDIT_TIER"  |"TOTAL_POLICIES"  |"AVG_CAR_AGE"  |"AVG_KILOMETERS"  |"TOTAL_CAR_VALUE"  |"AVG_DEDUCTIBLE"  |"UPDATED_AT"                      |
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|CUST_002456    |64     |19                |0                 |581             |32.90         |0.0000                 |Poor           |0                 |NULL           |NULL              |NULL               

In [10]:
online_config = OnlineConfig(enable=True, target_lag="1 minute")

customer_fv = FeatureView(
    name="CUSTOMER_RISK_FEATURES",
    entities=[customer_entity],
    feature_df=customer_features_df,
    timestamp_col="UPDATED_AT",
    refresh_freq="5 minutes",
    refresh_mode="AUTO",
    desc="Customer risk and profile features for insurance pricing with online serving",
    online_config=online_config
)

customer_fv = fs.register_feature_view(
    feature_view=customer_fv,
    version="v1"
)

print("Feature View 'CUSTOMER_RISK_FEATURES' registered with Online Feature Store enabled")
print(f"Online target lag: 1 minute")
print(f"Offline refresh freq: 5 minutes")

/Users/ccarrero/opt/anaconda3/envs/py311_env/lib/python3.11/site-packages/snowflake/ml/feature_store/feature_store.py:2700: UserWarning: Your pipeline won't be incrementally refreshed due to: "Query contains the aggregation function on a float-typed expression as projection with join in same query block, which is not supported for change tracking. Please consider replacing the floating point expression with a fixed-point number.".
  self._check_dynamic_table_refresh_mode(feature_view_name)


Feature View 'CUSTOMER_RISK_FEATURES' registered with Online Feature Store enabled
Online target lag: 1 minute
Offline refresh freq: 5 minutes


In [11]:
print("Registered Feature Views:")
fs.list_feature_views().to_pandas()

Registered Feature Views:


,NAME,VERSION,DATABASE_NAME,SCHEMA_NAME,CREATED_ON,OWNER,DESC,ENTITIES,REFRESH_FREQ,REFRESH_MODE,SCHEDULING_STATE,WAREHOUSE,CLUSTER_BY,ONLINE_CONFIG
0,CUSTOMER_RISK_FEATURES,v1,CC_ML_INSURANCE,CAR_PRICING,2026-01-28 23:11:21.916,SPCS_PSE_ROLE,Customer risk and profile features for insuran...,"[\n ""CUSTOMER""\n]",5 minutes,FULL,ACTIVE,COMPUTE_WH,"[""CUSTOMER_ID"", ""UPDATED_AT""]","{""enable"": true, ""target_lag"": ""1 minute"", ""re..."


## 4. Prepare Training Dataset

In [12]:
spine_df = (
    session.table(f"{DATABASE}.{SCHEMA}.POLICIES")
    .join(
        session.table(f"{DATABASE}.{SCHEMA}.CUSTOMERS").select("CUSTOMER_ID", "GENDER", "STATE"),
        on="CUSTOMER_ID"
    )
    .with_column("CAR_AGE", F.lit(current_year_val) - F.col("CAR_YEAR"))
    .select(
        "CUSTOMER_ID",
        "CAR_MAKE",
        "CAR_MODEL",
        "CAR_AGE",
        "KILOMETERS",
        "ENGINE_SIZE",
        "FUEL_TYPE",
        "TRANSMISSION",
        "COVERAGE_TYPE",
        "ESTIMATED_CAR_VALUE",
        "GENDER",
        "STATE",
        "ANNUAL_PREMIUM",
        F.col("UPDATED_AT").cast("timestamp").alias("TS")
    )
)

print(f"Spine DataFrame created with {spine_df.count()} rows")
print("Spine columns (join key + policy data + label):")
print(spine_df.columns)
spine_df.show(5)

Spine DataFrame created with 8000 rows
Spine columns (join key + policy data + label):
['CUSTOMER_ID', 'CAR_MAKE', 'CAR_MODEL', 'CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE', 'ESTIMATED_CAR_VALUE', 'GENDER', 'STATE', 'ANNUAL_PREMIUM', 'TS']
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CAR_MAKE"  |"CAR_MODEL"  |"CAR_AGE"  |"KILOMETERS"  |"ENGINE_SIZE"  |"FUEL_TYPE"  |"TRANSMISSION"  |"COVERAGE_TYPE"  |"ESTIMATED_CAR_VALUE"  |"GENDER"  |"STATE"  |"ANNUAL_PREMIUM"  |"TS"                 |
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|CUST_004018    |BMW         |3 Series     |8      

In [13]:
training_dataset = fs.generate_dataset(
    name=f"{DATABASE}.{SCHEMA}.CAR_INSURANCE_TRAINING_DATASET",
    spine_df=spine_df,
    features=[customer_fv],
    spine_timestamp_col="TS",
    spine_label_cols=["ANNUAL_PREMIUM"],
    desc="Training dataset for car insurance premium prediction with customer risk features"
)

print("Snowflake Dataset created successfully!")
print(f"Dataset name: {training_dataset.fully_qualified_name}")

training_dataset_df = training_dataset.read.to_snowpark_dataframe()
print(f"Training dataset rows: {training_dataset_df.count()}")
training_dataset_df.show(5)

Snowflake Dataset created successfully!
Dataset name: CC_ML_INSURANCE.CAR_PRICING.CAR_INSURANCE_TRAINING_DATASET
Training dataset rows: 8000
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CAR_MAKE"  |"CAR_MODEL"  |"CAR_AGE"  |"KILOMETERS"  |"ENGINE_SIZE"       |"FUEL_TYPE"  |"TRANSMISSION"  |"COVERAGE_TYPE"  |"ESTIMATED_CAR_VALUE"  |"GENDER"  |"STATE"  |"ANNUAL_PREMIUM"    |"TS"                 |"AGE"  |"YEARS_LICENSED"  |"CLAIMS_HISTORY"  |"CREDIT_SCORE"  |"RISK_SCORE"        |"AVG_CLAIMS_PER_YEAR"  |"CREDIT_TIER"  |"TOTAL_POLICIES"  |"AVG_CAR_AGE"  |"AVG_KILOMETERS"  |"TOTA

In [14]:
training_df = training_dataset_df.to_pandas()
print(f"Training DataFrame shape: {training_df.shape}")
print("\nDataset Statistics:")
training_df.describe()

Training DataFrame shape: (8000, 26)

Dataset Statistics:


,CAR_AGE,KILOMETERS,ENGINE_SIZE,ESTIMATED_CAR_VALUE,ANNUAL_PREMIUM,TS,AGE,YEARS_LICENSED,CLAIMS_HISTORY,CREDIT_SCORE,RISK_SCORE,AVG_CLAIMS_PER_YEAR,TOTAL_POLICIES,AVG_CAR_AGE,AVG_KILOMETERS,TOTAL_CAR_VALUE,AVG_DEDUCTIBLE
count,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000,8000.000000
mean,8.140750,122923.875000,2.868338,11258.202707,1474.999241,2026-01-29 08:10:56.608875008,41.967125,10.832875,0.395250,699.687375,42.932213,0.110629,2.617250,8.140756,122923.970875,29574.018495,1057.338250
min,0.000000,1000.000000,1.500000,965.260010,400.000000,2026-01-29 08:10:55,18.000000,1.000000,0.000000,460.000000,0.000000,0.000000,1.000000,0.000000,1000.000000,965.260010,250.000000
25%,4.000000,47923.500000,2.000000,2569.239990,954.882523,2026-01-29 08:10:56,32.000000,3.000000,0.000000,648.000000,30.000000,0.000000,2.000000,6.000000,78810.500000,10286.860352,750.000000
50%,8.000000,109812.500000,2.500000,6014.750000,1331.544983,2026-01-29 08:10:57,42.000000,8.000000,0.000000,701.000000,39.000000,0.000000,2.000000,8.000000,118118.000000,23639.560547,1000.000000
75%,12.000000,181711.250000,3.500000,16478.915527,1824.120056,2026-01-29 08:10:57,52.000000,16.000000,1.000000,753.000000,53.400002,0.047600,3.000000,10.500000,162247.000000,41417.088867,1375.000000
max,16.000000,507701.000000,5.000000,59800.000000,5000.000000,2026-01-29 08:10:58,80.000000,62.000000,6.000000,850.000000,162.000000,5.000000,8.000000,16.000000,424422.000000,171547.765625,2000.000000
std,4.923918,88861.009653,1.076178,12070.060133,710.746694,NaN,14.006135,9.608654,0.757693,76.719140,19.977289,0.351471,1.257159,3.491677,62385.938125,24706.217770,452.571137


## 5. Feature Engineering and Model Training

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

categorical_cols = ['CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE', 'GENDER', 'STATE']
numeric_cols = ['CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE',
                'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE',
                'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE', 
                'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE']

label_encoders = {}
training_encoded = training_df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    training_encoded[f"{col}_ENCODED"] = le.fit_transform(training_encoded[col].astype(str))
    label_encoders[col] = le

feature_cols = numeric_cols + [f"{col}_ENCODED" for col in categorical_cols]

X = training_encoded[feature_cols]
y = training_encoded['ANNUAL_PREMIUM']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nFeatures: {feature_cols}")

Training set: (6400, 22)
Test set: (1600, 22)

Features: ['CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE', 'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE', 'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE', 'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE', 'CAR_MAKE_ENCODED', 'CAR_MODEL_ENCODED', 'FUEL_TYPE_ENCODED', 'TRANSMISSION_ENCODED', 'COVERAGE_TYPE_ENCODED', 'GENDER_ENCODED', 'STATE_ENCODED']


In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=feature_cols, index=X_test.index)

print("StandardScaler applied")
print(f"Scaled feature means (should be ~0): {X_train_scaled.mean().mean():.6f}")
print(f"Scaled feature stds (should be ~1): {X_train_scaled.std().mean():.6f}")

StandardScaler applied
Scaled feature means (should be ~0): -0.000000
Scaled feature stds (should be ~1): 1.000078


In [17]:
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_scaled, 
    y_train,
    eval_set=[(X_test_scaled, y_test)],
    verbose=20
)

print("\nXGBoost model trained!")

[0]	validation_0-rmse:707.85742
[20]	validation_0-rmse:313.54467
[40]	validation_0-rmse:229.74770
[60]	validation_0-rmse:208.82794
[80]	validation_0-rmse:201.78779
[100]	validation_0-rmse:200.56183
[120]	validation_0-rmse:200.65291
[140]	validation_0-rmse:201.95691
[160]	validation_0-rmse:202.30870
[180]	validation_0-rmse:203.05156
[199]	validation_0-rmse:203.79323

XGBoost model trained!


In [18]:
y_pred = xgb_model.predict(X_test_scaled)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Model Performance Metrics:")
print(f"  RMSE: ${rmse:.2f}")
print(f"  MAE:  ${mae:.2f}")
print(f"  R2:   {r2:.4f}")

metrics = {
    "rmse": float(rmse),
    "mae": float(mae),
    "r2": float(r2)
}

Model Performance Metrics:
  RMSE: $203.79
  MAE:  $144.06
  R2:   0.9274


In [19]:
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 10 Most Important Features:")
feature_importance.head(10)

Top 10 Most Important Features:


,feature,importance
19,COVERAGE_TYPE_ENCODED,0.428502
8,RISK_SCORE,0.197169
4,AGE,0.073304
15,CAR_MAKE_ENCODED,0.056426
5,YEARS_LICENSED,0.046462
7,CREDIT_SCORE,0.044563
14,AVG_DEDUCTIBLE,0.030804
6,CLAIMS_HISTORY,0.028343
3,ESTIMATED_CAR_VALUE,0.024661
9,AVG_CLAIMS_PER_YEAR,0.012716


## 6. Create Custom Model for Registry

In [20]:
from snowflake.ml.model import custom_model

class CarInsurancePricingModel(custom_model.CustomModel):
    
    def __init__(self, context: custom_model.ModelContext) -> None:
        super().__init__(context)
        import xgboost as xgb
        import pickle
        
        with open(context.path("xgb_model.ubj"), "rb") as f:
            self.model = pickle.load(f)
        with open(context.path("scaler.pkl"), "rb") as f:
            self.scaler = pickle.load(f)
        with open(context.path("label_encoders.pkl"), "rb") as f:
            self.label_encoders = pickle.load(f)
        with open(context.path("feature_cols.pkl"), "rb") as f:
            self.feature_cols = pickle.load(f)
    
    @custom_model.inference_api
    def predict(self, input_df: pd.DataFrame) -> pd.DataFrame:
        import numpy as np
        
        df = input_df.copy()
        
        categorical_cols = ['CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 
                           'COVERAGE_TYPE', 'GENDER', 'STATE']
        
        for col in categorical_cols:
            if col in df.columns:
                le = self.label_encoders[col]
                df[f"{col}_ENCODED"] = df[col].apply(
                    lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else 0
                )
        
        available_cols = [c for c in self.feature_cols if c in df.columns]
        X = df[available_cols]
        X_scaled = self.scaler.transform(X)
        
        predictions = self.model.predict(X_scaled)
        predictions = np.maximum(predictions, 400)
        predictions = np.minimum(predictions, 5000)
        
        return pd.DataFrame({"PREDICTED_PREMIUM": predictions})

print("Custom model class defined")

Custom model class defined


In [21]:
import tempfile
import pickle
import os as os_module

model_artifacts_dir = tempfile.mkdtemp()

with open(os_module.path.join(model_artifacts_dir, "xgb_model.ubj"), "wb") as f:
    pickle.dump(xgb_model, f)

with open(os_module.path.join(model_artifacts_dir, "scaler.pkl"), "wb") as f:
    pickle.dump(scaler, f)

with open(os_module.path.join(model_artifacts_dir, "label_encoders.pkl"), "wb") as f:
    pickle.dump(label_encoders, f)

with open(os_module.path.join(model_artifacts_dir, "feature_cols.pkl"), "wb") as f:
    pickle.dump(feature_cols, f)

print(f"Model artifacts saved to: {model_artifacts_dir}")
print(f"Files: {os_module.listdir(model_artifacts_dir)}")

Model artifacts saved to: /var/folders/40/5phm8bjx3nv3bhwrtt6p345c0000gn/T/tmppkb_v_vw
Files: ['scaler.pkl', 'label_encoders.pkl', 'feature_cols.pkl', 'xgb_model.ubj']


In [22]:
from snowflake.ml.model.model_signature import infer_signature

input_cols = ['CAR_AGE', 'KILOMETERS', 'ENGINE_SIZE', 'ESTIMATED_CAR_VALUE',
              'AGE', 'YEARS_LICENSED', 'CLAIMS_HISTORY', 'CREDIT_SCORE',
              'RISK_SCORE', 'AVG_CLAIMS_PER_YEAR', 'TOTAL_POLICIES', 'AVG_CAR_AGE', 
              'AVG_KILOMETERS', 'TOTAL_CAR_VALUE', 'AVG_DEDUCTIBLE',
              'CAR_MAKE', 'CAR_MODEL', 'FUEL_TYPE', 'TRANSMISSION', 'COVERAGE_TYPE',
              'GENDER', 'STATE']

signature = infer_signature(
    input_data=training_dataset_df.select(input_cols).limit(100),
    output_data=training_dataset_df.select("ANNUAL_PREMIUM").with_column_renamed("ANNUAL_PREMIUM", "PREDICTED_PREMIUM").limit(100)
)

print("Model signature inferred from training dataset")
print(signature)

Model signature inferred from training dataset
ModelSignature(
                    inputs=[
                        FeatureSpec(dtype=DataType.INT64, name='CAR_AGE', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='KILOMETERS', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='ENGINE_SIZE', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='ESTIMATED_CAR_VALUE', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='AGE', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='YEARS_LICENSED', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='CLAIMS_HISTORY', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='CREDIT_SCORE', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='RISK_SCORE', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='AVG_CLAIMS_PER_YEAR', nullable=True),
		FeatureSpec(dtype=DataType.INT64, name='TOTAL_POLICIES', nullable=True),
		FeatureSpec(dtype=DataType.DOUBLE, name='AVG_CAR_AGE', nullable=True),
		Fea

## 7. Register Model to Snowflake Model Registry

In [23]:
from snowflake.ml.registry import Registry
from snowflake.ml.model import type_hints

registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

print(f"Registry opened: {DATABASE}.{SCHEMA}")

Registry opened: CC_ML_INSURANCE.CAR_PRICING


In [24]:
model_context = custom_model.ModelContext(
    artifacts={
        "xgb_model.ubj": os_module.path.join(model_artifacts_dir, "xgb_model.ubj"),
        "scaler.pkl": os_module.path.join(model_artifacts_dir, "scaler.pkl"),
        "label_encoders.pkl": os_module.path.join(model_artifacts_dir, "label_encoders.pkl"),
        "feature_cols.pkl": os_module.path.join(model_artifacts_dir, "feature_cols.pkl")
    }
)

pricing_model = CarInsurancePricingModel(model_context)

test_sample = training_dataset_df.select(input_cols).limit(5).to_pandas()
test_pred = pricing_model.predict(test_sample)
print("Local model test predictions:")
print(test_pred)

Local model test predictions:
   PREDICTED_PREMIUM
0        1185.504761
1        2600.022217
2         518.542969
3        1687.048828
4        2709.802246


In [25]:
mv = registry.log_model(
    pricing_model,
    model_name="CAR_INSURANCE_PRICING_MODEL",
    version_name="v2",
    signatures={"predict": signature},
    sample_input_data=training_dataset_df.select(input_cols).limit(100),
    conda_dependencies=["xgboost", "scikit-learn", "pandas", "numpy"],
    comment="XGBoost model for car insurance premium prediction with StandardScaler preprocessing",
    metrics=metrics,
    options={
        "relax_version": True
    }
)

print(f"Model registered: {mv.model_name} version {mv.version_name}")
print(f"Metrics: {mv.show_metrics()}")

Model logged successfully.: 100%|██████████| 6/6 [00:39<00:00,  6.62s/it]                          
Model registered: CAR_INSURANCE_PRICING_MODEL version V2
Metrics: {'rmse': 203.79323074708157, 'mae': 144.06450553894044, 'r2': 0.9274322764180482}


In [26]:
print("Registered models:")
registry.show_models()

Registered models:


,created_on,name,model_type,database_name,schema_name,comment,owner,default_version_name,versions,aliases
0,2026-01-28 23:13:07.444000-08:00,CAR_INSURANCE_PRICING_MODEL,USER_MODEL,CC_ML_INSURANCE,CAR_PRICING,None,SPCS_PSE_ROLE,V2,"[""V2""]","{""DEFAULT"":""V2"",""FIRST"":""V2"",""LAST"":""V2""}"


## 8. Test Model Predictions

In [27]:
def calculate_risk_features_for_new_customer(age, years_licensed, credit_score=700):
    """Calculate risk features for new customers (same logic as Feature View)"""
    claims_history = 0
    risk_score = (
        (30 if age < 25 else (20 if age > 65 else 0)) +
        (claims_history * 15) +
        max(0, 40 - years_licensed) +
        max(0, (700 - credit_score) / 10)
    )
    avg_claims_per_year = 0.0
    credit_tier = ('Excellent' if credit_score >= 750 else 
                   'Good' if credit_score >= 700 else 
                   'Fair' if credit_score >= 650 else 'Poor')
    return {
        'CLAIMS_HISTORY': claims_history,
        'CREDIT_SCORE': credit_score,
        'RISK_SCORE': round(risk_score, 2),
        'AVG_CLAIMS_PER_YEAR': round(avg_claims_per_year, 4),
        'CREDIT_TIER': credit_tier,
        'AVG_DEDUCTIBLE': 500
    }

def prepare_inference_data(customer_id, car_data, fs, session):
    """
    Prepare data for inference:
    - Existing customer: fetch features from Online Feature Store
    - New customer: calculate features with defaults
    """
    if customer_id:
        try:
            customer_features = fs.retrieve_feature_values(
                spine_df=session.create_dataframe([(customer_id,)], schema=["CUSTOMER_ID"]),
                features=[customer_fv]
            ).to_pandas()
            
            if len(customer_features) > 0:
                for col in customer_features.columns:
                    if col != 'CUSTOMER_ID':
                        car_data[col] = customer_features[col].values[0]
                print(f"✓ Found existing customer {customer_id} - using Online Feature Store data")
                return car_data
        except Exception as e:
            print(f"Customer {customer_id} not found in Feature Store: {e}")
    
    print("→ New customer - using default values for risk features")
    risk_features = calculate_risk_features_for_new_customer(
        car_data['AGE'], car_data['YEARS_LICENSED']
    )
    car_data.update(risk_features)
    car_data['TOTAL_POLICIES'] = 0
    car_data['AVG_CAR_AGE'] = car_data['CAR_AGE']
    car_data['AVG_KILOMETERS'] = car_data['KILOMETERS']
    car_data['TOTAL_CAR_VALUE'] = car_data['ESTIMATED_CAR_VALUE']
    return car_data

print("Test Case 1: EXISTING CUSTOMER (using Online Feature Store)")
existing_customer_data = {
    'CUSTOMER_ID': 'CUST_000001',
    'CAR_AGE': 3, 'KILOMETERS': 30000, 'ENGINE_SIZE': 2.0,
    'ESTIMATED_CAR_VALUE': 25000, 'AGE': 35, 'YEARS_LICENSED': 17,
    'CAR_MAKE': 'Toyota', 'CAR_MODEL': 'Camry', 'FUEL_TYPE': 'Gasoline',
    'TRANSMISSION': 'Automatic', 'COVERAGE_TYPE': 'Standard', 'GENDER': 'M', 'STATE': 'CA'
}

print("\nTest Case 2: NEW CUSTOMER (using defaults - no claims, avg credit)")
new_customer_data = {
    'CUSTOMER_ID': None,
    'CAR_AGE': 1, 'KILOMETERS': 5000, 'ENGINE_SIZE': 3.0,
    'ESTIMATED_CAR_VALUE': 55000, 'AGE': 22, 'YEARS_LICENSED': 4,
    'CAR_MAKE': 'BMW', 'CAR_MODEL': '3 Series', 'FUEL_TYPE': 'Gasoline',
    'TRANSMISSION': 'Automatic', 'COVERAGE_TYPE': 'Premium', 'GENDER': 'M', 'STATE': 'NY'
}

test_cases_raw = [existing_customer_data.copy(), new_customer_data.copy()]
test_cases_list = []

for case in test_cases_raw:
    customer_id = case.pop('CUSTOMER_ID')
    prepared = prepare_inference_data(customer_id, case, fs, session)
    test_cases_list.append(prepared)

test_cases = pd.DataFrame(test_cases_list)

for col in categorical_cols:
    le = label_encoders[col]
    test_cases[f"{col}_ENCODED"] = test_cases[col].apply(
        lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else 0
    )

print("\nPrepared Test Cases (note: CLAIMS_HISTORY, CREDIT_SCORE, AVG_DEDUCTIBLE from Feature Store or defaults):")
test_cases[['CAR_MAKE', 'AGE', 'CLAIMS_HISTORY', 'CREDIT_SCORE', 'RISK_SCORE', 'AVG_DEDUCTIBLE', 'COVERAGE_TYPE']]

Test Case 1: EXISTING CUSTOMER (using Online Feature Store)

Test Case 2: NEW CUSTOMER (using defaults - no claims, avg credit)
✓ Found existing customer CUST_000001 - using Online Feature Store data
→ New customer - using default values for risk features

Prepared Test Cases (note: CLAIMS_HISTORY, CREDIT_SCORE, AVG_DEDUCTIBLE from Feature Store or defaults):


,CAR_MAKE,AGE,CLAIMS_HISTORY,CREDIT_SCORE,RISK_SCORE,AVG_DEDUCTIBLE,COVERAGE_TYPE
0,Toyota,49,0,688,22.2,750,Standard
1,BMW,22,0,700,66.0,500,Premium


In [28]:
predictions = pricing_model.predict(test_cases)

results = pd.DataFrame({
    'Scenario': ['Existing Customer (Feature Store)', 'New Customer (Calculated)'],
    'Description': [
        '35yo, Toyota Camry, 0 claims, Standard',
        '22yo, BMW 3-Series, 2 claims, Premium (High Risk)'
    ],
    'Risk Score': test_cases['RISK_SCORE'].values,
    'Predicted Premium': predictions['PREDICTED_PREMIUM'].values.round(2)
})

print("Predictions:")
results

Predictions:


,Scenario,Description,Risk Score,Predicted Premium
0,Existing Customer (Feature Store),"35yo, Toyota Camry, 0 claims, Standard",22.2,951.039978
1,New Customer (Calculated),"22yo, BMW 3-Series, 2 claims, Premium (High Risk)",66.0,3225.280029


## 9. Deploy Model as Inference Service (SPCS)

In [29]:
COMPUTE_POOL_NAME = "CC_INFERENCE_CPU_POOL"

session.sql(f"""
CREATE COMPUTE POOL IF NOT EXISTS {COMPUTE_POOL_NAME}
    MIN_NODES = 1
    MAX_NODES = 4
    INSTANCE_FAMILY = CPU_X64_S
    AUTO_RESUME = TRUE
    AUTO_SUSPEND_SECS = 300
""").collect()

print(f"Compute pool {COMPUTE_POOL_NAME} created/verified")

compute_pools = session.sql("SHOW COMPUTE POOLS").collect()
print("\nAvailable Compute Pools:")
for pool in compute_pools:
    print(f"  - {pool['name']}: {pool['instance_family']}, State: {pool['state']}")

Compute pool CC_INFERENCE_CPU_POOL created/verified

Available Compute Pools:
  - AUDIO_PROCESSING_CP_DATA_DOWNLOAD: CPU_X64_M, State: SUSPENDED
  - AUDIO_PROCESSING_CP_GPU_NV_S_5_NODES: GPU_NV_S, State: SUSPENDED
  - BACKEND_CPU_POOL: CPU_X64_S, State: ACTIVE
  - CCARRERO_CPU_POOL: CPU_X64_S, State: SUSPENDED
  - CC_INFERENCE_CPU_POOL: CPU_X64_S, State: SUSPENDED
  - CPU_L: CPU_X64_L, State: SUSPENDED
  - CPU_M: CPU_X64_M, State: SUSPENDED
  - CPU_S: CPU_X64_S, State: SUSPENDED
  - CPU_S_3: CPU_X64_S, State: SUSPENDED
  - CPU_S_HM: HIGHMEM_X64_S, State: SUSPENDED
  - CPU_XS: CPU_X64_XS, State: SUSPENDED
  - CPU_XS_3: CPU_X64_XS, State: SUSPENDED
  - CPU_XS_INFERENCE: CPU_X64_S, State: SUSPENDED
  - CPU_XS_INFERENCE2: CPU_X64_S, State: SUSPENDED
  - DEMO_POOL: CPU_X64_S, State: SUSPENDED
  - DEMO_POOL_CPU: CPU_X64_S, State: SUSPENDED
  - DETECTION_TRAINING_GPU: GPU_NV_M, State: SUSPENDED
  - FASTAPI_POOL: CPU_X64_M, State: SUSPENDED
  - GPU_NV_M_COMPUTE_POOL: GPU_NV_M, State: SUSPENDED

In [30]:
SERVICE_NAME = "CAR_INSURANCE_INFERENCE_SVC"

try:
    existing = session.sql(f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
    if existing:
        print(f"Dropping existing service: {SERVICE_NAME}")
        session.sql(f"DROP SERVICE IF EXISTS {DATABASE}.{SCHEMA}.{SERVICE_NAME}").collect()
except:
    pass

print(f"Ready to create service: {SERVICE_NAME}")
print(f"Using compute pool: {COMPUTE_POOL_NAME}")

Ready to create service: CAR_INSURANCE_INFERENCE_SVC
Using compute pool: CC_INFERENCE_CPU_POOL


In [31]:
mv.create_service(
    service_name=SERVICE_NAME,
    service_compute_pool=COMPUTE_POOL_NAME,
    ingress_enabled=True,
    min_instances = 1,
    max_instances = 4,
    autocapture=True
)

print(f"Service '{SERVICE_NAME}' creation initiated")
print("Note: Service creation takes 5-15 minutes to complete")

create_service logs saved to: /Users/ccarrero/Library/Logs/snowflake-ml/model_deploy_fa87d355_1769670795.log       
To see logs in console, set log level to INFO: logging.getLogger().setLevel(logging.INFO)                     
Model service created successfully: 100%|██████████| 6/6 [04:43<00:00, 47.30s/it]                             
Service 'CAR_INSURANCE_INFERENCE_SVC' creation initiated
Note: Service creation takes 5-15 minutes to complete


In [32]:
import time

print("Checking service status...")
for i in range(10):
    status = session.sql(f"SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
    if status:
        state = status[0]['status']
        print(f"  Attempt {i+1}: Service state = {state}")
        if state == 'RUNNING':
            print("\nService is RUNNING!")
            break
    else:
        print(f"  Attempt {i+1}: Service not found yet")
    time.sleep(30)
else:
    print("\nService still starting. Check status later with:")
    print(f"  SHOW SERVICES LIKE '{SERVICE_NAME}' IN SCHEMA {DATABASE}.{SCHEMA};")

Checking service status...
  Attempt 1: Service state = RUNNING

Service is RUNNING!


In [33]:
#session.sql(f"ALTER SERVICE {SERVICE_NAME} SET MIN_INSTANCES = 1").collect()

In [34]:
print("Model Services:")
mv.list_services()

Model Services:


,name,status,inference_endpoint,internal_endpoint,autocapture_enabled
0,CC_ML_INSURANCE.CAR_PRICING.CAR_INSURANCE_INFE...,RUNNING,None,http://car-insurance-inference-svc.lvpw.svc.sp...,True


## 10. Create Gateway for Stable URL Access

A Gateway provides a stable URL that routes to SPCS endpoints. Unlike direct service endpoints that change when services are recreated, the Gateway URL remains constant, making it ideal for production integrations.

In [35]:
role_name = session.get_current_role()
print (f'Current role: {role_name}')

session.sql(f'GRANT SERVICE ROLE {DATABASE}.{SCHEMA}.{SERVICE_NAME}!ALL_ENDPOINTS_USAGE TO ROLE {role_name}').collect()


Current role: "SPCS_PSE_ROLE"


[Row(status='Statement executed successfully.')]

In [36]:
GATEWAY_NAME = "CAR_INSURANCE_GATEWAY"

gateway_spec = f"""
CREATE OR REPLACE GATEWAY {GATEWAY_NAME}
  FROM SPECIFICATION $$
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: {DATABASE}.{SCHEMA}.{SERVICE_NAME}!inference
    weight: 100
$$;
"""

print("Creating Gateway...")
print(f"Gateway Name: {GATEWAY_NAME}")
print(f"Target Service: {DATABASE}.{SCHEMA}.{SERVICE_NAME}")
print(f"\nSQL:\n{gateway_spec}")

session.sql(gateway_spec).collect()
print("\nGateway created successfully!")

Creating Gateway...
Gateway Name: CAR_INSURANCE_GATEWAY
Target Service: CC_ML_INSURANCE.CAR_PRICING.CAR_INSURANCE_INFERENCE_SVC

SQL:

CREATE OR REPLACE GATEWAY CAR_INSURANCE_GATEWAY
  FROM SPECIFICATION $$
spec:
  type: traffic_split
  split_type: custom
  targets:
  - type: endpoint
    value: CC_ML_INSURANCE.CAR_PRICING.CAR_INSURANCE_INFERENCE_SVC!inference
    weight: 100
$$;


Gateway created successfully!


In [37]:
print("Verifying Gateway...")
gateway_info = session.sql(f"DESCRIBE GATEWAY {GATEWAY_NAME}").collect()
print("\nGateway Details:")
for row in gateway_info:
    print(f"  {row}")



Verifying Gateway...

Gateway Details:
  Row(name='CAR_INSURANCE_GATEWAY', ingress_url='Endpoints provisioning in progress... check back in a few minutes', database_name='CC_ML_INSURANCE', schema_name='CAR_PRICING', owner='SPCS_PSE_ROLE', owner_role_type='ROLE', spec='---\n\nspec:\n  type: traffic_split\n  split_type: custom\n  targets:\n  - type: endpoint\n    value: CC_ML_INSURANCE.CAR_PRICING.CAR_INSURANCE_INFERENCE_SVC!inference\n    weight: 100\n', created_on=datetime.datetime(2026, 1, 28, 23, 18, 1, 929000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>), updated_on=datetime.datetime(2026, 1, 28, 23, 18, 2, 137000, tzinfo=<DstTzInfo 'America/Los_Angeles' PST-1 day, 16:00:00 STD>), comment=None)


In [38]:
gateway_info = session.sql(f"DESCRIBE GATEWAY {GATEWAY_NAME}").collect()
if gateway_info:
    gateway_url = f"https://{gateway_info[0]['ingress_url']}"
    print("="*60)
    print("GATEWAY ACCESS INFORMATION")
    print("="*60)
    print(f"\nGateway Name: {GATEWAY_NAME}")
    print(f"Gateway URL:  {gateway_url}")
    print(f"\nTarget Endpoint: {DATABASE}.{SCHEMA}.{SERVICE_NAME}!predict")
    print("\nUsage Example (Python):")
    print(f'''
import requests
import json

gateway_url = "{gateway_url}"
headers = {{
    "Authorization": "Snowflake Token=\\"<YOUR_JWT_TOKEN>\\"",
    "Content-Type": "application/json"
}}

data = {{
    "data": [[
        35,           # age
        "Male",       # gender  
        2.0,          # years_since_claim
        "Sedan",      # vehicle_type
        12000,        # annual_mileage
        750,          # credit_score
        5,            # years_as_customer
        10,           # vehicle_age
        "Comprehensive", # coverage_type
        "Urban",      # location
        1             # has_previous_claim
    ]]
}}

response = requests.post(gateway_url, headers=headers, json=data)
prediction = response.json()
print(f"Predicted Premium: ${{prediction['data'][0][0]:.2f}}")
''')
else:
    print("Gateway not found. Please check the gateway was created successfully.")

GATEWAY ACCESS INFORMATION

Gateway Name: CAR_INSURANCE_GATEWAY
Gateway URL:  https://Endpoints provisioning in progress... check back in a few minutes

Target Endpoint: CC_ML_INSURANCE.CAR_PRICING.CAR_INSURANCE_INFERENCE_SVC!predict

Usage Example (Python):

import requests
import json

gateway_url = "https://Endpoints provisioning in progress... check back in a few minutes"
headers = {
    "Authorization": "Snowflake Token=\"<YOUR_JWT_TOKEN>\"",
    "Content-Type": "application/json"
}

data = {
    "data": [[
        35,           # age
        "Male",       # gender  
        2.0,          # years_since_claim
        "Sedan",      # vehicle_type
        12000,        # annual_mileage
        750,          # credit_score
        5,            # years_as_customer
        10,           # vehicle_age
        "Comprehensive", # coverage_type
        "Urban",      # location
        1             # has_previous_claim
    ]]
}

response = requests.post(gateway_url, headers=headers, json=d

## 11. Summary

### Created Resources:
- **Database/Schema**: `CC_ML_INSURANCE.CAR_PRICING`
- **Tables**: `CUSTOMERS` (5,000 rows), `POLICIES` (8,000 rows)
- **Feature Store**: `CUSTOMER_RISK_FEATURES` with Online Feature Store enabled
- **Model**: `CAR_INSURANCE_PRICING_MODEL` (XGBoost with StandardScaler)
- **Service**: `CAR_INSURANCE_INFERENCE_SVC` (SPCS real-time inference)
- **Gateway**: `CAR_INSURANCE_GATEWAY` (Stable URL for API access)

### Benefits of Gateway:
- **Stable URL**: Gateway URL never changes, even if service is recreated
- **Traffic Splitting**: Can route to multiple endpoints with weighted distribution
- **Failover**: Automatic failover if primary endpoint becomes unhealthy

### Next Steps:
1. Wait for service to be READY
2. Use Gateway URL for production API integrations
3. Deploy Streamlit app to test the full pipeline

In [39]:
print("="*60)
print("CAR INSURANCE ML PIPELINE - SUMMARY")
print("="*60)
print(f"\nDatabase: {DATABASE}")
print(f"Schema: {SCHEMA}")
print(f"\nTables:")
print(f"  - CUSTOMERS: {session.table('CUSTOMERS').count()} rows")
print(f"  - POLICIES: {session.table('POLICIES').count()} rows")
print(f"\nFeature Store:")
print(f"  - Entity: CUSTOMER")
print(f"  - Feature View: CUSTOMER_RISK_FEATURES (Online enabled)")
print(f"\nModel:")
print(f"  - Name: CAR_INSURANCE_PRICING_MODEL")
print(f"  - Version: v1")
print(f"  - Type: XGBoost Regressor")
print(f"  - Preprocessing: StandardScaler + LabelEncoder")
print(f"  - RMSE: ${metrics['rmse']:.2f}")
print(f"  - R2: {metrics['r2']:.4f}")
print(f"\nInference Service:")
print(f"  - Name: {SERVICE_NAME}")
print(f"  - Compute Pool: {COMPUTE_POOL_NAME}")
print(f"\nGateway (Stable API Access):")
print(f"  - Name: {GATEWAY_NAME}")
gateway_result = session.sql(f"SHOW GATEWAYS LIKE '{GATEWAY_NAME}' IN SCHEMA {DATABASE}.{SCHEMA}").collect()
if gateway_result:
    print(f"  - URL: {gateway_result[0]['ingress_url']}")
print("="*60)

CAR INSURANCE ML PIPELINE - SUMMARY

Database: CC_ML_INSURANCE
Schema: CAR_PRICING

Tables:
  - CUSTOMERS: 5000 rows
  - POLICIES: 8000 rows

Feature Store:
  - Entity: CUSTOMER
  - Feature View: CUSTOMER_RISK_FEATURES (Online enabled)

Model:
  - Name: CAR_INSURANCE_PRICING_MODEL
  - Version: v1
  - Type: XGBoost Regressor
  - Preprocessing: StandardScaler + LabelEncoder
  - RMSE: $203.79
  - R2: 0.9274

Inference Service:
  - Name: CAR_INSURANCE_INFERENCE_SVC
  - Compute Pool: CC_INFERENCE_CPU_POOL

Gateway (Stable API Access):
  - Name: CAR_INSURANCE_GATEWAY


KeyError: 'ingress_url'